# Instagram Post Engagement Prediction (Google Colab)

This notebook implements a machine learning pipeline to predict Instagram post engagement using multimodal features (text embeddings from captions and hashtags, and structured numerical features) in Google Colab. The caption column is cleaned to remove hashtags listed in the hashtags column to avoid redundancy. The dataset is split into training, validation, and test sets before merging with sentiment data to prevent leakage. The final merged dataset and test split are saved for manual verification.

## 1. Setup and Install Libraries

**Instructions:**
- Upload `data.csv` and `sentiment.csv` to Colab using the file upload cell below.
- Outputs (e.g., `final_features.csv`, `test_features.csv`) are saved in `/content/` and can be downloaded or saved to Google Drive.

In [ ]:
# Install required libraries
!pip install pandas numpy sentence-transformers scikit-learn xgboost shap

## 2. Upload Data Files

In [ ]:
from google.colab import files
import pandas as pd
import numpy as np
import re

# Upload data.csv and sentiment.csv
print("Please upload data.csv and sentiment.csv")
uploaded = files.upload()

# Load datasets
data_df = pd.read_csv('/content/data.csv')
sentiment_df = pd.read_csv('/content/sentiment.csv')

## 3. Load and Split Data

**Splitting Strategy:**
- Split `data.csv` into training (70%), validation (15%), and test (15%) sets before merging with `sentiment.csv` to prevent data leakage.
- Use stratified splitting on `engagement_binary` to maintain class balance.
- Merge each split with `sentiment.csv` separately.

In [ ]:
from sklearn.model_selection import train_test_split

# Split data into train, validation, and test sets
train_df, temp_df = train_test_split(data_df, test_size=0.3, random_state=42, stratify=data_df['engagement_binary'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['engagement_binary'])

# Merge each split with sentiment data
train_merged = pd.merge(train_df, sentiment_df, on='post_id', how='left')
val_merged = pd.merge(val_df, sentiment_df, on='post_id', how='left')
test_merged = pd.merge(test_df, sentiment_df, on='post_id', how='left')

# Drop rows with missing target
train_merged = train_merged.dropna(subset=['engagement_rate', 'engagement_binary'])
val_merged = val_merged.dropna(subset=['engagement_rate', 'engagement_binary'])
test_merged = test_merged.dropna(subset=['engagement_rate', 'engagement_binary'])

# Save test split for manual verification
test_merged.to_csv('/content/test_split.csv', index=False)
print(f"Test split saved as '/content/test_split.csv' for manual verification ({len(test_merged)} rows)")
files.download('/content/test_split.csv')

# Display split sizes
print(f"Train set: {len(train_merged)} rows")
print(f"Validation set: {len(val_merged)} rows")
print(f"Test set: {len(test_merged)} rows")

## 4. Data Preprocessing

**Best Practices:**
- Clean captions by removing hashtags listed in the hashtags column to avoid redundancy.
- Clean hashtags and impute missing values with "no_caption" or "no_hashtags".
- Scale numerical features, impute missing values with median, encode categorical variables.
- Apply identical preprocessing to train, validation, and test sets.

In [ ]:
from sklearn.preprocessing import StandardScaler

# Clean text columns
def clean_text(text, hashtags=None):
    if pd.isna(text):
        return "no_caption" if hashtags is not None else "no_hashtags"
    text = text.lower().strip()
    text = ''.join(c for c in text if c.isalnum() or c.isspace() or c in '#@')
    if hashtags is not None:
        # Remove hashtags from caption that appear in hashtags column
        if pd.notna(hashtags):
            hashtag_list = hashtags.split(', ')
            for hashtag in hashtag_list:
                text = re.sub(rf'\b{re.escape(hashtag)}\b', '', text, flags=re.IGNORECASE)
            text = ' '.join(text.split())  # Remove extra spaces
        if not text:
            text = "no_caption"
    return text

# Apply cleaning to captions and hashtags
for df in [train_merged, val_merged, test_merged]:
    df['caption'] = df.apply(lambda x: clean_text(x['caption'], x['hashtags']), axis=1)
    df['hashtags'] = df['hashtags'].apply(lambda x: clean_text(x))

# Encode categorical variables
categorical_cols = ['media_type', 'Category']
for df in [train_merged, val_merged, test_merged]:
    df = pd.get_dummies(df, columns=categorical_cols, drop_first=True, inplace=True)

# Numerical features
numerical_cols = [
    'likes', 'comments_count', '#Followers', '#Followees', '#Posts',
    'caption_length', 'num_hashtags', 'sentiment_positive', 'sentiment_negative',
    'sentiment_neutral', 'avg_comment_sentiment', 'sentiment_weighted_engagement'
]

# Impute and scale numerical features
scaler = StandardScaler()
train_merged[numerical_cols] = scaler.fit_transform(train_merged[numerical_cols])
val_merged[numerical_cols] = scaler.transform(val_merged[numerical_cols])
test_merged[numerical_cols] = scaler.transform(test_merged[numerical_cols])

# Save processed dataframes
train_merged.to_csv('/content/train_processed.csv', index=False)
val_merged.to_csv('/content/val_processed.csv', index=False)
test_merged.to_csv('/content/test_processed.csv', index=False)
print("Processed dataframes saved in /content/")
files.download('/content/train_processed.csv')
files.download('/content/val_processed.csv')
files.download('/content/test_processed.csv')

## 5. Generate Text Embeddings

**Best Practices:**
- Use Sentence-BERT (all-MiniLM-L6-v2) for 384-dimensional embeddings.
- Generate embeddings for cleaned captions (without hashtags) and hashtags separately.
- Process each split independently to maintain separation.

In [ ]:
from sentence_transformers import SentenceTransformer

# Load Sentence-BERT model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Generate embeddings for each split
def generate_embeddings(df, text_column):
    embeddings = model.encode(df[text_column].tolist(), batch_size=32, show_progress_bar=True)
    return pd.DataFrame(embeddings, columns=[f'{text_column}_emb_{i}' for i in range(embeddings.shape[1])], index=df.index)

# Train set embeddings
train_caption_emb = generate_embeddings(train_merged, 'caption')
train_hashtag_emb = generate_embeddings(train_merged, 'hashtags')
train_features = pd.concat([train_merged[numerical_cols], train_caption_emb, train_hashtag_emb], axis=1)

# Validation set embeddings
val_caption_emb = generate_embeddings(val_merged, 'caption')
val_hashtag_emb = generate_embeddings(val_merged, 'hashtags')
val_features = pd.concat([val_merged[numerical_cols], val_caption_emb, val_hashtag_emb], axis=1)

# Test set embeddings
test_caption_emb = generate_embeddings(test_merged, 'caption')
test_hashtag_emb = generate_embeddings(test_merged, 'hashtags')
test_features = pd.concat([test_merged[numerical_cols], test_caption_emb, test_hashtag_emb], axis=1)

## 6. Save Final Merged Dataset

**Saving Strategy:**
- Combine train and validation features with targets into `final_features.csv` for training and tuning.
- Save test features with targets as `test_features.csv` for manual verification.
- Include numerical features, caption embeddings, hashtag embeddings, and targets.

In [ ]:
# Combine train and validation features with targets
final_features = pd.concat([train_features, val_features], axis=0)
final_targets = pd.concat([train_merged[['engagement_rate', 'engagement_binary']], 
                           val_merged[['engagement_rate', 'engagement_binary']]], axis=0)
final_df = pd.concat([final_features, final_targets], axis=1)

# Save final merged dataset
final_df.to_csv('/content/final_features.csv', index=False)
print("Final merged dataset saved as '/content/final_features.csv'")
files.download('/content/final_features.csv')

# Save test features with targets
test_df = pd.concat([test_features, test_merged[['engagement_rate', 'engagement_binary']]], axis=1)
test_df.to_csv('/content/test_features.csv', index=False)
print("Test features saved as '/content/test_features.csv' for manual verification")
files.download('/content/test_features.csv')

## 7. Model Selection and Training

**Model Selection:**
- Regression: XGBoost for `engagement_rate`.
- Classification: XGBoost for `engagement_binary`.
- Hyperparameter Tuning: Use validation set for tuning.

In [ ]:
import xgboost as xgb

# Define features and targets
X_train = train_features
y_train_reg = train_merged['engagement_rate']
y_train_clf = train_merged['engagement_binary']

X_val = val_features
y_val_reg = val_merged['engagement_rate']
y_val_clf = val_merged['engagement_binary']

X_test = test_features
y_test_reg = test_merged['engagement_rate']
y_test_clf = test_merged['engagement_binary']

# Train initial models
reg_model = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=100, max_depth=5, learning_rate=0.1)
reg_model.fit(X_train, y_train_reg, eval_set=[(X_val, y_val_reg)], early_stopping_rounds=10, verbose=False)

clf_model = xgb.XGBClassifier(objective='binary:logistic', n_estimators=100, max_depth=5, learning_rate=0.1)
clf_model.fit(X_train, y_train_clf, eval_set=[(X_val, y_val_clf)], early_stopping_rounds=10, verbose=False)

## 8. Hyperparameter Tuning

**Approach:**
- Use validation set to tune `max_depth`, `learning_rate`, and `n_estimators`.
- Select the best model based on validation performance.

In [ ]:
from sklearn.metrics import mean_squared_error, f1_score

# Manual tuning for regression
best_reg_model = None
best_val_mse = float('inf')
for max_depth in [3, 5, 7]:
    for lr in [0.05, 0.1, 0.2]:
        model = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=100, max_depth=max_depth, learning_rate=lr)
        model.fit(X_train, y_train_reg, eval_set=[(X_val, y_val_reg)], early_stopping_rounds=10, verbose=False)
        y_val_pred = model.predict(X_val)
        mse = mean_squared_error(y_val_reg, y_val_pred)
        if mse < best_val_mse:
            best_val_mse = mse
            best_reg_model = model

# Manual tuning for classification
best_clf_model = None
best_val_f1 = 0
for max_depth in [3, 5, 7]:
    for lr in [0.05, 0.1, 0.2]:
        model = xgb.XGBClassifier(objective='binary:logistic', n_estimators=100, max_depth=max_depth, learning_rate=lr)
        model.fit(X_train, y_train_clf, eval_set=[(X_val, y_val_clf)], early_stopping_rounds=10, verbose=False)
        y_val_pred = model.predict(X_val)
        f1 = f1_score(y_val_clf, y_val_pred)
        if f1 > best_val_f1:
            best_val_f1 = f1
            best_clf_model = model

## 9. Model Evaluation

**Testing Approach:**
- Cross-Validation: 5-fold CV on training data for stability.
- Test Set: Evaluate on test set for final performance.
- Metrics: MSE, R² for regression; Accuracy, Precision, Recall, F1 for classification.

In [ ]:
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score

# Cross-validation for regression
kf = KFold(n_splits=5, shuffle=True, random_state=42)
reg_cv_mse = []
for train_idx, val_idx in kf.split(X_train):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train_reg.iloc[train_idx], y_train_reg.iloc[val_idx]
    model = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=100, max_depth=5, learning_rate=0.1)
    model.fit(X_tr, y_tr)
    y_val_pred = model.predict(X_val)
    reg_cv_mse.append(mean_squared_error(y_val, y_val_pred))
print(f"Regression CV MSE: {np.mean(reg_cv_mse):.4f} ± {np.std(reg_cv_mse):.4f}")

# Cross-validation for classification
clf_cv_f1 = []
for train_idx, val_idx in kf.split(X_train):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train_clf.iloc[train_idx], y_train_clf.iloc[val_idx]
    model = xgb.XGBClassifier(objective='binary:logistic', n_estimators=100, max_depth=5, learning_rate=0.1)
    model.fit(X_tr, y_tr)
    y_val_pred = model.predict(X_val)
    clf_cv_f1.append(f1_score(y_val, y_val_pred))
print(f"Classification CV F1: {np.mean(clf_cv_f1):.4f} ± {np.std(clf_cv_f1):.4f}")

# Test set evaluation
y_test_pred_reg = best_reg_model.predict(X_test)
test_mse = mean_squared_error(y_test_reg, y_test_pred_reg)
test_r2 = r2_score(y_test_reg, y_test_pred_reg)
print(f"Regression Test - MSE: {test_mse:.4f}, R²: {test_r2:.4f}")

y_test_pred_clf = best_clf_model.predict(X_test)
test_accuracy = accuracy_score(y_test_clf, y_test_pred_clf)
test_precision = precision_score(y_test_clf, y_test_pred_clf)
test_recall = recall_score(y_test_clf, y_test_pred_clf)
test_f1 = f1_score(y_test_clf, y_test_pred_clf)
print(f"Classification Test - Accuracy: {test_accuracy:.4f}, Precision: {test_precision:.4f}, Recall: {test_recall:.4f}, F1: {test_f1:.4f}")

# Plot feature importance
xgb.plot_importance(best_reg_model, max_num_features=10)
import matplotlib.pyplot as plt
plt.show()

## 10. Model Interpretation

**Interpreting Outputs:**
- Use SHAP for feature contributions.
- Correlate hashtag embeddings with specific hashtags.

In [ ]:
import shap

# SHAP for regression model
explainer = shap.TreeExplainer(best_reg_model)
shap_values = explainer.shap_values(X_test)

# Summary plot
shap.summary_plot(shap_values, X_test, plot_type="bar", max_display=10)

# Detailed SHAP plot for a single prediction
shap.initjs()
shap.force_plot(explainer.expected_value, shap_values[0], X_test.iloc[0], matplotlib=True)

# Analyze high-impact hashtags
important_features = pd.Series(best_reg_model.feature_importances_, index=X_test.columns).nlargest(10)
hashtag_features = [f for f in important_features.index if 'hashtag_emb' in f]
print("Top Hashtag Embedding Features:", hashtag_features)

# Correlate hashtag embeddings with hashtags
hashtag_corr = pd.DataFrame(test_hashtag_emb, columns=[f'hashtag_emb_{i}' for i in range(test_hashtag_emb.shape[1])])
hashtag_corr['hashtags'] = test_merged['hashtags']
for feature in hashtag_features:
    top_hashtags = hashtag_corr.groupby('hashtags')[feature].mean().nlargest(5)
    print(f"Top hashtags for {feature}:\n", top_hashtags)

## 11. Suggestions for Improvement

- Add temporal features from timestamp (e.g., hour, day of week).
- Experiment with neural networks for complex interactions.
- Use topic modeling on captions/hashtags.
- Implement SMOTE for imbalanced engagement_binary.

## 12. Saving Models and Embeddings

In [ ]:
import joblib

# Save models
best_reg_model.save_model('/content/xgb_regression_model.json')
best_clf_model.save_model('/content/xgb_classification_model.json')
files.download('/content/xgb_regression_model.json')
files.download('/content/xgb_classification_model.json')

# Save embeddings
np.save('/content/train_caption_embeddings.npy', train_caption_emb.values)
np.save('/content/train_hashtag_embeddings.npy', train_hashtag_emb.values)
np.save('/content/val_caption_embeddings.npy', val_caption_emb.values)
np.save('/content/val_hashtag_embeddings.npy', val_hashtag_emb.values)
np.save('/content/test_caption_embeddings.npy', test_caption_emb.values)
np.save('/content/test_hashtag_embeddings.npy', test_hashtag_emb.values)
files.download('/content/train_caption_embeddings.npy')
files.download('/content/train_hashtag_embeddings.npy')
files.download('/content/val_caption_embeddings.npy')
files.download('/content/val_hashtag_embeddings.npy')
files.download('/content/test_caption_embeddings.npy')
files.download('/content/test_hashtag_embeddings.npy')

# Save scaler
joblib.dump(scaler, '/content/scaler.pkl')
files.download('/content/scaler.pkl')

## 13. Optional: Save to Google Drive

# Mount Google Drive (uncomment to use)
# from google.colab import drive
# drive.mount('/content/drive')

# Save files to Google Drive
# !cp /content/final_features.csv /content/drive/MyDrive/final_features.csv
# !cp /content/test_features.csv /content/drive/MyDrive/test_features.csv
# !cp /content/test_split.csv /content/drive/MyDrive/test_split.csv
# !cp /content/xgb_regression_model.json /content/drive/MyDrive/xgb_regression_model.json
# !cp /content/xgb_classification_model.json /content/drive/MyDrive/xgb_classification_model.json
# !cp /content/*.npy /content/drive/MyDrive/
# !cp /content/scaler.pkl /content/drive/MyDrive/scaler.pkl

## 14. Manual Verification

- Verify Test Split: Check `/content/test_split.csv` and `/content/test_features.csv` against `/content/train_processed.csv` and `/content/val_processed.csv` using `post_id`:

```python
train_ids = pd.read_csv('/content/train_processed.csv')['post_id']
test_ids = pd.read_csv('/content/test_split.csv')['post_id']
overlap = set(train_ids).intersection(set(test_ids))
print("Overlap between train and test:", len(overlap))  # Should be 0
```

- Verify Caption Cleaning: Inspect `/content/train_processed.csv` and `/content/test_processed.csv` to ensure hashtags are removed from caption.
- Final Merged Dataset: Check `/content/final_features.csv` for features and targets.
- Download Files: All saved files are automatically downloaded via `files.download()`. Check your local downloads folder.

## 15. Resources

- Sentence-BERT: [Hugging Face Sentence-Transformers](https://www.sbert.net/)
- XGBoost: [Official Documentation](https://xgboost.readthedocs.io/)
- SHAP: [SHAP Documentation](https://shap.readthedocs.io/)
- Feature Engineering: [Feature Engineering for Machine Learning](https://www.coursera.org/learn/feature-engineering)